# Day 47 — PyTorch basics: tensors, autograd, simple NN
Objectives:
- Build an MLP.
- Training/validation split.
- Evaluate accuracy.
Using sklearn make_classification for simplicity.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
X,y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
sc = StandardScaler(); X = sc.fit_transform(X).astype(np.float32)
Xtr,Xte,ytr,yte = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)
Xtr, Xte = torch.from_numpy(Xtr), torch.from_numpy(Xte)
ytr, yte = torch.from_numpy(ytr).long(), torch.from_numpy(yte).long()
model = nn.Sequential(nn.Linear(20,64), nn.ReLU(), nn.Linear(64,2))
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(50):
    opt.zero_grad(); out = model(Xtr); loss = loss_fn(out, ytr); loss.backward(); opt.step()
pred = model(Xte).argmax(1); acc = (pred==yte).float().mean().item(); acc


## Learner exercises and progressive hints

1. Add dropout to the MLP and compare results.
2. Implement a small minibatch training loop with `DataLoader`.

### Progressive hints

1. Place `Dropout(p=...)` after an activation. Compare multiple seeded runs or
   validation curves, because one final accuracy is noisy.
2. Wrap float32 features and long labels in a `TensorDataset`; shuffle only the
   training loader. Move `zero_grad`, forward, loss, backward, and step inside
   the batch loop.

The separate solution proceeds to early stopping, `StepLR`, and CUDA mixed
precision. Mixed precision is an optional GPU optimization; the code path must
remain correct on CPU.

### Additional mastery practice

Build a training loop whose dtype, shape, mode, randomness, and checkpoint state can be inspected and resumed on CPU without hidden notebook state.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

3. **Shape and dtype contract:** Write assertions at the start of a classification training step for feature shape/dtype, target shape/dtype, and logits shape.
   **Progressive hint:** CrossEntropyLoss expects floating logits `(batch, classes)` and integer class indices `(batch,)` with dtype long.
4. **Validation implementation:** Implement an evaluation function that returns sample-weighted loss and accuracy, restores the caller's prior train/eval mode, and never retains an autograd graph.
   **Progressive hint:** Remember `was_training = model.training`, call eval and no_grad, aggregate counts, then restore train mode only if it was previously active.
5. **DataLoader reproducibility:** Run two shuffled DataLoaders with the same seed and compare batch order. Then state what changes when using worker processes.
   **Progressive hint:** Pass a seeded `torch.Generator`; worker initialization and external NumPy/Python randomness need their own deliberate seeds.
6. **Portable checkpoint:** Save model, optimizer, epoch, metric history, and configuration, then reload on CPU and resume one step. Explain `state_dict` versus serializing the entire model object.
   **Progressive hint:** Save plain state dictionaries plus architecture/config metadata. Use `map_location='cpu'` and recreate the model class before loading.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.


In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 3 — Shape and dtype contract


# Practice 4 — Validation implementation


# Practice 5 — DataLoader reproducibility


# Practice 6 — Portable checkpoint
